## 10. 大模型预训练与微调
本章主要介绍了基于transformers库的预训练与微调，同时还介绍了人类对齐的一些方法。

### 10.1  基于transformers库训练模型
transformers库对模型训练做了很好的封装，使用该库提供的Trainer类可通过简单的三个步骤就完成模型训练。第一步是对数据进行预处理，这一步主要是对数据进行分割并对文本进行分词；第二步则是设置模型训练过程的超参数（Hyperparameter），这一步设置的数据对模型训练效果有较大影响；最后一步则是创建Trainer类的实例，并通过调用train方法开启模型训练。

#### 10.1.1  数据预处理
数据预处理包括加载数据集、数据分割、分词等步骤，下面的代码展示了拆分IMDB并进行分词的过程：

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# 加载数据集和分词器
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 数据预处理
def preprocess(data):
    return tokenizer(data['text'], padding=True, truncation=True)

train_dataset = dataset["train"].map(preprocess, batched=True)
test_dataset = dataset["test"].map(preprocess, batched=True)
print(train_dataset, test_dataset, sep="\n")

small_train_dataset = train_dataset.shuffle(seed=42).select(range(100))
small_eval_dataset = test_dataset.shuffle(seed=42).select(range(10))
small_test_dataset = test_dataset.shuffle(seed=24).select(range(10))
print(small_train_dataset, small_test_dataset, sep="\n")

#### 10.1.2  设置超参数
超参数包括学习率、优化器、批处理大小、训练轮数等等，transformers库为这些超参数抽象了一个类TrainingArguments，通过这个类可以直接为训练设置所有超参数，具体参数列表请参考书中表格。

#### 10.1.3  训练模型
大模型训练的正向传播和反向传播过程被封装在Trainer类中，只要创建了Trainer对象，调用其train方法就可以完成模型训练了。

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from evaluate import load
import numpy as np

# 加载一个未经过预训练的模型，随机初始化
config = AutoConfig.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_config(config)

# 设置超参数
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    eval_strategy="epoch",
)

# 加载评估指标
accuracy = load("accuracy")
f1 = load("f1")

# 定义评估函数
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(
            predictions=predictions, references=labels) | \
            f1.compute(predictions=predictions, references=labels)

# 初始化 Trainer
trainer = Trainer(
    model=model,  # 使用随机初始化的模型
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

# 从头开始训练
trainer.train()
# 评估模型
result = trainer.evaluate(eval_dataset=small_test_dataset)
print(result)

### 10.2  大模型分布式训练*
大模型分布式训练可分为数据并行和模型并行两个级别，前者一般仅应用于大模型预训练阶段，而后者也可在推理阶段中借鉴和使用。

#### 10.2.1  数据并行
数据并行的基本思想是在每个计算节点上加载模型的完整副本，每个副本针对不同数据分片并行地执行正向传播和反向传播。数据并行在PyTorch中分为分布式数据并行（Distributed Data Parallel，以下简称DDP）和完全分片数据并行（Fully Sharded Data Parallel，以下简称FSDP）两种，其中DDP是PyTorch原生支持的分布式训练方法，而FSDP则基本上是从FairScale中借鉴而来。具体请参考书中介绍。

#### 10.2.2  模型并行
模型并行可分为张量并行（Tensor Parallel，以下简称TP）和流水线并行（Pipeline Parallel，以下简称PP）两种类型，前者是将单层内部的计算分解到多个计算节点上，而后者则是将模型中不同的层拆分到不同的计算节点上。具体请参考书中介绍。

这一节的内容比较难懂，但对于不需要训练模型的情况并不需要掌握，所以可以大致阅读一下，在需要时再回过头来精读。

### 10.3  大模型微调
大模型微调的方法有很多，以不同标准来划分可得到完全不同的类别。如果按微调修改参数的数量来划分，微调可分为全量微调（Full Fine-tuning）和部分微调（Partial Fine-tuning）两类。全量单任务微调最为简单，它与预训练的代码几乎没有不同。部分微调有多种方法，比较简单就是只针对模型的“头部”进行微调。除了冻结模型主干以外，参数高效微调（Parameter Efficient Fine-Tuning，以下简称PEFT）也是非常重要的部分微调方法。由于PEFT简单而高效，它已经逐渐成为模型微调的主流方法。

#### 10.3.1  低秩自适应微调
低秩自适应（Low Rank Adaptation，以下简称LoRA）核心思想是冻结预训练模型原参数，通过训练额外的低秩矩阵达到微调目的。

![LoRA结构](./images/lora.png)

#### 10.3.2 软提示微调
软提示微调通过调整输入模型的向量达到微调的效果。这类微调方法本质上还是提示工程，只不过提示的模板不是由人类设计出来的，而是由模型通过微调自主学习到的。软提示微调可分为P微调、前缀微调和提示微调三种，它们之间的差异并不是特别大，详细请参考书中介绍。

#### 10.3.3 使用PEFT库
Hugging Face为了支持PEFT，专门开发了与transformers、evaluate等平级的peft库。PEFT库支持LoRA、软提示等多种微调方法，采用模型包装的形式添加PEFT功能。peft同样依赖于TrainingArguments和Trainer配置微调，但在加载模型后需要再使用peft.get_peft_model将模型包装起来，即：
```python
from peft import LoraConfig, TaskType, get_peft_model

model = ... # 加载模型
peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, 
                           inference_mode=False, r=8, 
                           lora_alpha=32, lora_dropout=0.1)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
```

在使用PEFT微调后的模型做推理时，依然需要通过peft.get_peft_model对原模型做包装。下面的代码展示了使用该方法加载PEFT模型做推理的全过程：

In [ ]:
from peft import PeftConfig, PeftModel, get_peft_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 加载预训练BERT模型
mn = "google-bert/bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(mn)
m = AutoModelForSequenceClassification.from_pretrained(mn, num_labels=5)
# 从路径./peft/checkpoint-4加载PEFT模型
m = PeftModel.from_pretrained(m, "./peft/checkpoint-4")

# 通过PeftConfig和get_peft_model加载PEFT模型
# config = PeftConfig.from_pretrained("./peft/checkpoint-4")
# print(config)
# m = get_peft_model(model, config)
# m.load_adapter("./peft/checkpoint-4", "lora")

m.eval()
inputs = tokenizer("I love this book", return_tensors="pt")
output = m(**inputs)
print(output)

### 10.4  大模型的人类对齐**
如果说微调可以让大模型学会执行不同类型或不同领域的任务，那么人类对齐（Human Alignment）则是教会大模型遵循人类的价值观、期望和目标。第2章在介绍生成式AI项目周期时也曾提到过，人类对齐应让大模型的行为符合3H原则，即有用性（Helpful）、诚实性（Honesty）和无害性（Harmless）。有不少可以让大模型遵从3H原则的优化技术，它们大都与基于人类反馈的强化学习（Reinforcement Learning from Human Feedback，以下简称RLHF）有些关联。顾名思义，RLHF是采用强化学习优化模型的方法，它本质上也是对模型的一种微调，只不过更侧重于人类偏好的对齐。所以想要理解RLHF，还得先掌握一些强化学习的基础知识。

这一节的内容比较难懂，并且在实践中很少会用到，读者可根据情况自行选择是否阅读。

### 10.5  本章小节
本章介绍了大模型预训练、微调以及人类对齐的相关实现技术，对于没有深度学习和强化基础的初学者来说有一定难度。但好在transformers库对模型训练进行了非常好的封装，只要准备好数据和模型并通过TrainingArguments设置好训练参数，通过Trainer的train方法就可以开启模型训练了。大模型的预训练和微调都可以基于这两个类进行，人类对齐虽然没有直接使用这两个类，但编写代码的流程与模式大同小异。如果读者对本章列举各种算法不甚理解也没有关系，只要掌握通过transformers库训练模型的基本方法就可以了。后续随着时间的推移和经验的积累，这些复杂的算法和公式都会慢慢理解。

大模型预训练与普通神经网络的训练没有本质区别，但由于大模型的参数规模巨大，所以必须借助分布式技术加载模型并加速训练。大模型的分布式训练分为数据并行和模型并行两种，数据并行对训练数据进行分片，但模型在处理数据时仍然是针对所有参数进行的；模型并行不仅会将模型参数分片，在处理数据时也只针对分片后的参数进行运算。数据并行包括分布式数据并行DDP和完全分片数据并行FSDP两种，二者都会对训练数据进行分片，区别在于DDP会在每个GPU上加载完整的模型副本，而FSDP则会将模型分片保存在不同的GPU上。模型并行分为张量并行TP和流水线并行PP两种，前者可直接将参数矩阵拆分到不同GPU上，而后者则从整体结构上对模型进行拆分。

大模型微调可分为全量微调和部分微调两种，二者的区别在于它们调整参数的范围不同。全量微调调整模型所有参数，而部分微调则只调整少量参数。最简单的部分微调就是冻结模型主干参数，而只调整模型头部参数，这适用于采用头机制的大语言模型。PEFT一般通过给模型附加额外的可训练组件达到微调的目的，由于额外的可训练组件远小于大模型，所以PEFT微调效率更为高效。低秩自适应微调LoRA是PEFT中最为重要的微调方法之一，它通过在前馈神经网络中附加低秩参数矩阵实现微调目标。低秩矩阵参数量不仅更小而且可通过秩来控制，所以从效率和灵活性上都非常不错的选择。软提示微调是另一种PEFT方法，它通过在输入中附加额外的可学习向量实现微调。软提示微调分P-Tuning、Prefix Tuning和Prompt Tuning三种，其中P-Tuning和Prompt Tuning适用于NLU任务的微调，而Prefix Tuning则适用于NLG任务的微调。Hugging Face专门开发了peft库用于参数高效微调，采用与包装模型的形式实现PEFT微调，但微调过程仍然基于Trainer和TrainingArguments进行。

本章还介绍了大模型对齐人类偏好的RLHF和DPO等微调方法。其中，RLHF是较早出现的对齐方法之一，DPO则可以认为是对RLHF的优化方法。RLHF需要先由人工对大模型生成的数据做人类偏好标注，然后再使用这些数据训练独立的奖励模型。奖励模型可以对大模型生成的内容进行评分，目的是取代人类在RLHF中的作用。基于奖励模型给出的评分，就可通过强化学习和PPO算法优化大模型了。DPO则将大模型自身看成是奖励模型，通过对比大模型与人类偏好的差异来计算误差和优化模型。最终DPO算法可以在不借助奖励模型和强化学习的情况下，通过一般的模型训练方法就实现人类偏好对齐。另一种对PPO算法的优化是GRPO，它将PPO算法中的单个输出扩展为多个，并使用这些输出的平均收益取代价值模型的作用，进而减少了GRPO在存储与运算上的需求。GRPO仍然属于RLHF的范畴，需要独立的奖励模型参与训练。
总的来看，本章介绍的内容都是让模型获取知识或技能的训练方法，只是在训练方法、资源消耗和侧重点等诸多方面上有所不同。大模型预训练一般在海量数据集上进行的无监督训练，资源消耗巨大成本也非常高，它侧重于让模型形成适用于各类任务的通用知识；微调是在少量有标签的数据集进行的有监督训练，资源消耗相对要小很多，侧重于让模型学会特定领域或特定任务的知识；人类对齐则是在人类偏好数据集上进行的微调，可以基于强化学习也可以是普通的有监督训练，侧重于让模型的生成偏好与人类相一致。这些方法组合在一起，共同构建起了大模型的知识体系和任务技能，是当今大模型智能水平得以不断演进的技术基础。